# CSE 151B — Full public evaluation

Live in **`notebooks/full_public.ipynb`**. Same inference stack as [`dev.ipynb`](dev.ipynb), run on all of **`data/public.jsonl`** (1126 rows).

1. (**Colab A100**) `%pip` → restart runtime → Drive copy cell.
2. Set **`MAX_TOKENS`** in §2 (default 16k → **pub-002**).
3. Prompts are **per question**: baseline for MCQ and single-blank free-form; §1.3 multi-blank format when `[ANS]` count > 1.
4. Generate with checkpointing → score → summary + topic breakdown.

Artifacts: `results/full_public_{Nk}.jsonl` (and `.responses.jsonl` checkpoint). Baseline reference: [pub-001](../docs/log/runs/pub-001-full-public-8k.md).

### Google Colab

**Colab (A100):** run the `%pip` cell below, restart runtime, continue. **Local:** use your venv — same packages (`vllm`, `transformers`, …).

> **Note:** `bitsandbytes` is not needed — Qwen3-4B fits in native bf16 on A100 (~8 GB), which is faster than quantized loading.


In [ ]:
# Colab: skip when working locally with an existing venv.
%pip install -q sympy numpy tqdm "antlr4-python3-runtime==4.11.1"
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu129 --upgrade
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.20.0/vllm-0.20.0+cu129-cp38-abi3-manylinux_2_31_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip install -q transformers

## 2. Imports & configuration

- `MAX_TOKENS` — 8192 = pub-001 path; **16384** = pub-002 path
- `RUN_ID` — registry id; auto-derived when `None`
- `OUTPUT_PATH` — `results/full_public_{Nk}.jsonl`
- `DATA_PATH` — `data/public.jsonl`

Prompts are chosen automatically in §4 (no variant knob).

In [ ]:
import json
import os
from pathlib import Path


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    if (here / "data").is_dir():
        return here
    if here.name == "notebooks" and (here.parent / "data").is_dir():
        return here.parent
    up = here.parent
    if (up / "data").is_dir():
        return up
    return here


REPO_ROOT = _repo_root()

MAX_TOKENS = 16384
RUN_ID = None  # auto: pub-002, pub-001

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID   = "0"
DATA_PATH = str(REPO_ROOT / "data/public.jsonl")

_TOKEN_K = MAX_TOKENS // 1024
if RUN_ID is None:
    if MAX_TOKENS >= 16384:
        RUN_ID = "pub-002"
    elif MAX_TOKENS <= 8192:
        RUN_ID = "pub-001"
    else:
        RUN_ID = f"pub-{_TOKEN_K}k"
OUTPUT_PATH = str(REPO_ROOT / f"results/full_public_{_TOKEN_K}k.jsonl")

print(f"REPO_ROOT   = {REPO_ROOT}")
print(f"RUN_ID      = {RUN_ID}")
print(f"MAX_TOKENS  = {MAX_TOKENS} ({_TOKEN_K}k)")
print(f"OUTPUT_PATH = {OUTPUT_PATH}")

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import glob
import site

def _prepend_nvidia_libs_to_ld_path() -> None:
    roots = list(site.getsitepackages())
    user_site = getattr(site, "getusersitepackages", lambda: None)()
    if user_site:
        roots.append(user_site)
    libdirs: list[str] = []
    seen: set[str] = set()
    for root in roots:
        for d in glob.glob(os.path.join(root, "nvidia", "*", "lib")):
            if os.path.isdir(d) and d not in seen:
                seen.add(d)
                libdirs.append(d)
    if libdirs:
        sep = os.pathsep
        os.environ["LD_LIBRARY_PATH"] = sep.join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")]).strip(sep)


_prepend_nvidia_libs_to_ld_path()

import contextlib
import io
import re
import sys
from typing import Optional

@contextlib.contextmanager
def _jupyter_stdout_for_vllm():
    try:
        sys.stdout.fileno()
    except (io.UnsupportedOperation, AttributeError, OSError):
        saved_out, saved_err = sys.stdout, sys.stderr
        dup1, dup2 = os.dup(1), os.dup(2)
        try:
            sys.stdout = os.fdopen(dup1, "w", buffering=1)
            sys.stderr = os.fdopen(dup2, "w", buffering=1)
            yield
        finally:
            sys.stdout.close()
            sys.stderr.close()
            sys.stdout, sys.stderr = saved_out, saved_err
    else:
        yield

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

### Colab: put `public.jsonl` on Google Drive, then copy it here

The Colab VM does not include your git repo. Upload **`public.jsonl`** to  
`My Drive/CSE151B/data/public.jsonl` (or edit `DRIVE_JSONL` in the next cell), run the next cell, then load the dataset.


In [ ]:
import shutil

try:
    from google.colab import drive
except ImportError:
    print("Skip: not Google Colab — use repo `data/public.jsonl`.")
else:
    drive.mount("/content/drive")
    DRIVE_JSONL = Path("/content/drive/MyDrive/CSE151B/data/public.jsonl")
    DRIVE_PROJECT_ROOT = DRIVE_JSONL.parent.parent
    if not DRIVE_JSONL.is_file():
        raise FileNotFoundError(
            f"Missing on Drive: {DRIVE_JSONL}\n"
            "Upload public.jsonl or set DRIVE_JSONL."
        )
    (REPO_ROOT / "data").mkdir(parents=True, exist_ok=True)
    dest = REPO_ROOT / "data/public.jsonl"
    shutil.copy2(DRIVE_JSONL, dest)
    print(f"Copied to {dest.resolve()}")

In [ ]:
def n_ans_placeholders(question: str) -> int:
    return question.count("[ANS]")


data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options") for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form) from {DATA_PATH}")

mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))
multi_sample = next(
    (d for d in data if not d.get("options") and n_ans_placeholders(d["question"]) > 1),
    free_sample,
)

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2)[:1200], "...\n" if len(json.dumps(mcq_sample)) > 1200 else "\n")
print("── Free-form sample ──")
print(json.dumps(free_sample, indent=2)[:1200], "...\n" if len(json.dumps(free_sample)) > 1200 else "\n")
if multi_sample is not free_sample:
    print(f"── Multi-blank sample ({n_ans_placeholders(multi_sample['question'])} blanks) ──")
    print(json.dumps(multi_sample, indent=2)[:1200], "...\n" if len(json.dumps(multi_sample)) > 1200 else "\n")

## 4. Prompt construction

Per question — no global variant switch:

| Case | System prompt |
|------|----------------|
| MCQ | baseline |
| Free-form, 1 `[ANS]` | baseline |
| Free-form, 2+ `[ANS]` | §1.3 multi-blank (`\boxed{a}, \boxed{b}, ...`, judger-compatible) |

In [ ]:
_MATH_BASELINE = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

_MCQ_BASELINE = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

_MATH_MULTI_BLANK = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "For problems with multiple [ANS] placeholders, put each sub-answer in its own "
    "\\boxed{}, separated by commas, in the order the blanks appear "
    "(e.g. \\boxed{3}, \\boxed{7}). Do not use labels like 'Answer 1:' between boxes. "
    "For single-answer problems, use one \\boxed{}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return _MCQ_BASELINE, f"{question}\n\nOptions:\n{opts_text}"
    n_blanks = n_ans_placeholders(question)
    if n_blanks > 1:
        example = ", ".join("\\boxed{...}" for _ in range(n_blanks))
        user = (
            f"{question}\n\n"
            f"The problem has {n_blanks} [ANS] blanks. "
            f"After your reasoning, give {n_blanks} comma-separated \\boxed{{}} values "
            f"in order: {example}"
        )
        return _MATH_MULTI_BLANK, user
    return _MATH_BASELINE, question


def prompt_mode(question: str, options: Optional[list]) -> str:
    if options:
        return "mcq/baseline"
    return "multi-blank" if n_ans_placeholders(question) > 1 else "baseline"


for label, item in [
    ("MCQ", mcq_sample),
    ("Free-form (single-blank)", free_sample),
    *( [(f"Multi-blank ({n_ans_placeholders(multi_sample['question'])} blanks)", multi_sample)] if multi_sample is not free_sample else [] ),
]:
    mode = prompt_mode(item["question"], item.get("options"))
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} [{mode}] ──")
    print(f"  system: {sys_p[:120]}...")
    print(f"  user (first 300 chars): {usr_p[:300]}...\n")

## 5. Load model with vLLM (A100 profile)

Same **bf16** profile as `dev.ipynb` — not the starter L4 INT8 path. See [`docs/infra/vllm-inference-config.md`](../docs/infra/vllm-inference-config.md).

| Parameter | Value |
|-----------|-------|
| `dtype` | `bfloat16` |
| `max_model_len` | 32768 |
| `enable_prefix_caching` | True |
| `enable_chunked_prefill` | True |

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

with _jupyter_stdout_for_vllm():
    llm = LLM(
        model=MODEL_ID,
        dtype="bfloat16",
        enable_prefix_caching=True,
        gpu_memory_utilization=0.92,
        max_model_len=32768,
        trust_remote_code=True,
        max_num_seqs=128,
        max_num_batched_tokens=32768,
        enable_chunked_prefill=True,
    )

sampling_params_free = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)


def make_mcq_sampling_params(n_options: int) -> SamplingParams:
    return SamplingParams(
        max_tokens=MAX_TOKENS,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
        min_p=0.0,
    )


print("Model loaded.")

## 6. Generate responses

Checkpoint: `results/full_public_{Nk}.responses.jsonl` (Drive on Colab). Delete checkpoint to regenerate from scratch.

In [ ]:
CHUNK_SIZE = 100

try:
    _results_dir = DRIVE_PROJECT_ROOT / "results"
except NameError:
    _results_dir = Path(OUTPUT_PATH).parent

_results_dir.mkdir(parents=True, exist_ok=True)
response_checkpoint = _results_dir / (Path(OUTPUT_PATH).stem + ".responses.jsonl")
print(f"Checkpoint path: {response_checkpoint}")

completed: dict = {}
if response_checkpoint.exists():
    with open(response_checkpoint) as f:
        for line in f:
            r = json.loads(line)
            completed[r["id"]] = r["response"]
    print(f"Resumed from checkpoint: {len(completed)} responses already generated")

remaining = [d for d in data if d.get("id") not in completed]
print(f"Generating {len(remaining)} remaining / {len(data)} total...")

for chunk_start in range(0, len(remaining), CHUNK_SIZE):
    chunk = remaining[chunk_start : chunk_start + CHUNK_SIZE]

    prompts = []
    chunk_params = []
    for item in chunk:
        system, user = build_prompt(item["question"], item.get("options"))
        prompt_text = tokenizer.apply_chat_template(
            [{"role": "system", "content": system},
             {"role": "user",   "content": user}],
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append(prompt_text)
        opts = item.get("options")
        chunk_params.append(
            make_mcq_sampling_params(len(opts)) if opts else sampling_params_free
        )

    with _jupyter_stdout_for_vllm():
        outputs = llm.generate(prompts, sampling_params=chunk_params)

    with open(response_checkpoint, "a") as f:
        for item, out in zip(chunk, outputs):
            response = out.outputs[0].text.strip()
            completed[item["id"]] = response
            f.write(json.dumps({"id": item["id"], "response": response}) + "\n")

    done = len(completed)
    print(f"  {done}/{len(data)}  ({done / len(data) * 100:.1f}%)")

responses = [completed[d["id"]] for d in data]
print(f"Done. {len(responses)} responses ready.")

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [ ]:
_judger_override = os.environ.get("CSE151B_JUDGER_DIR", "").strip()
try:
    _drive_root = str(DRIVE_PROJECT_ROOT)
except NameError:
    _drive_root = ""
JUDGER_ROOT = _judger_override or _drive_root or str(REPO_ROOT)

print(f"JUDGER_ROOT={JUDGER_ROOT}")
sys.path.insert(0, os.path.abspath(JUDGER_ROOT))
from judger import Judger
judger = Judger(strict_extract=False)

In [ ]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

## 8. Summary

Prints metrics for [`docs/log/experiments.md`](../docs/log/experiments.md). Default run: **pub-002** (16k, adaptive prompts).

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

_q_by_id = {d["id"]: d["question"] for d in data}
multi_free  = [r for r in free_res if n_ans_placeholders(_q_by_id[r["id"]]) > 1]
single_free = [r for r in free_res if r not in multi_free]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

mcq_pct   = acc(mcq_res)
free_pct  = acc(free_res)
multi_pct = acc(multi_free)
overall_pct = acc(results)
n_total = len(results)

print("=" * 50)
print("FULL PUBLIC RESULTS")
print("=" * 50)
print(f"  RUN_ID       : {RUN_ID}")
print(f"  MAX_TOKENS   : {MAX_TOKENS}")
print(f"  Prompts      : baseline (MCQ + single-blank); multi-blank when 2+ [ANS]")
print(f"  N            : {n_total}  ({len(mcq_res)} MCQ, {len(free_res)} free-form)")
print("-" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({mcq_pct:.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({free_pct:.2f}%)")
if multi_free:
    print(f"  Multi-blank: {sum(r['correct'] for r in multi_free):4d} / {len(multi_free):4d}  ({multi_pct:.2f}%)")
if single_free:
    print(f"  Single-blank: {sum(r['correct'] for r in single_free):4d} / {len(single_free):4d}  ({acc(single_free):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {n_total:4d}  ({overall_pct:.2f}%)")
print("=" * 50)

print(f"\nArtifact  : {Path(OUTPUT_PATH).name}")
print(f"Registry  : docs/log/runs/{RUN_ID}-*.md")

In [ ]:
import re
from collections import defaultdict

TOPIC_PATTERNS = [
    ("sequences/recurrences", re.compile(r"sequence|recurrence|series|arithmetic|geometric|fibonacci", re.I)),
    ("geometry",              re.compile(r"triangle|circle|angle|polygon|area|perimeter|volume|surface|congruent|similar|parallelogram|trapezoid|chord|radius|diameter", re.I)),
    ("limits",               re.compile(r"\blimit\b|lim_|\\lim", re.I)),
    ("linear algebra",        re.compile(r"matrix|determinant|eigenvalue|eigenvector|vector space|rank|linear transform", re.I)),
    ("derivatives",          re.compile(r"derivative|differentiat|\\frac\{d|d/dx", re.I)),
    ("integration",          re.compile(r"integral|integrat|antiderivative|\\int", re.I)),
    ("probability/stats",    re.compile(r"probability|expected value|variance|distribution|random variable|combinat|permutation", re.I)),
    ("number theory",        re.compile(r"prime|divisib|modulo|congruence|gcd|lcm|diophantine", re.I)),
    ("polynomials/algebra",  re.compile(r"polynomial|quadratic|cubic|root|factor|expand|simplify|equation", re.I)),
]

def classify_topic(question: str) -> str:
    for name, pat in TOPIC_PATTERNS:
        if pat.search(question):
            return name
    return "other"

id_to_q = {d['id']: d['question'] for d in data}

topic_correct  = defaultdict(int)
topic_total    = defaultdict(int)
topic_mcq_c    = defaultdict(int)
topic_mcq_t    = defaultdict(int)

for r in results:
    q = id_to_q.get(r['id'], '')
    t = classify_topic(q)
    topic_total[t]   += 1
    topic_correct[t] += int(r['correct'])
    if r['is_mcq']:
        topic_mcq_t[t] += 1
        topic_mcq_c[t] += int(r['correct'])

rows = sorted(topic_total.keys(), key=lambda t: topic_correct[t] / topic_total[t])

print(f"{'Topic':<25} {'N':>5} {'Acc':>7}  {'MCQ N':>6} {'MCQ Acc':>8}")
print("-" * 60)
for t in rows:
    n         = topic_total[t]
    topic_acc = topic_correct[t] / n * 100
    mn        = topic_mcq_t[t]
    ma        = topic_mcq_c[t] / mn * 100 if mn else float('nan')
    print(f"{t:<25} {n:>5} {topic_acc:>6.1f}%  {mn:>6} {ma:>7.1f}%")
print("-" * 60)
total_acc = sum(r['correct'] for r in results) / len(results) * 100
print(f"{'TOTAL':<25} {len(results):>5} {total_acc:>6.1f}%")

# Save topic breakdown to Drive results folder
try:
    _topic_out = DRIVE_PROJECT_ROOT / "results" / (Path(OUTPUT_PATH).stem + "_topics.json")
except NameError:
    _topic_out = Path(OUTPUT_PATH).parent / (Path(OUTPUT_PATH).stem + "_topics.json")

_topic_data = {
    t: {
        "n": topic_total[t],
        "correct": topic_correct[t],
        "accuracy": round(topic_correct[t] / topic_total[t] * 100, 2),
        "mcq_n": topic_mcq_t[t],
        "mcq_correct": topic_mcq_c[t],
        "mcq_accuracy": round(topic_mcq_c[t] / topic_mcq_t[t] * 100, 2) if topic_mcq_t[t] else None,
    }
    for t in topic_total
}
_topic_out.parent.mkdir(parents=True, exist_ok=True)
with open(_topic_out, 'w') as _f:
    json.dump(_topic_data, _f, indent=2)
print(f"Topic breakdown saved to {_topic_out.resolve()}")

## 9. Save results

**Where files go**

- **Google Colab** (after Drive setup): `My Drive/CSE151B/results/` — filename from `OUTPUT_PATH` (default `full_public_16k.jsonl` for pub-002).
- **Local:** `results/full_public_{Nk}.jsonl` under `REPO_ROOT`.

**With evaluation** (public set): each line `{id, is_mcq, gold, response, correct}`.

Toggle `SAVE_EVAL` below.

In [ ]:
SAVE_EVAL = True

try:
    out_path = DRIVE_PROJECT_ROOT / "results" / Path(OUTPUT_PATH).name
except NameError:
    out_path = Path(OUTPUT_PATH)

out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path.resolve()}")

In [ ]:
try:
    from google.colab import runtime
    runtime.unassign()
except ImportError:
    print("Not running in Colab — skipping runtime termination.")